Might want to re-rank the list so the most relevant items are at the top of the list 

Hence not polluting the context

This increases precision 

this all adds latency and costs 

we use managed re rankers for this

`cohere.com`

get an api key and add to env 
`CO_API_KEY=`

`uv add --dev cohere`

In [21]:
import openai
from pydantic import BaseModel, Field
import pandas as pd
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadFieldSchema, PointStruct, Document, Prefetch, FusionQuery, PayloadSchemaType

In [22]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [23]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Retrieval

In [24]:
query = "can i get a tablet"

In [25]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [30]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid",
        prefetch=[
            Prefetch(
                query=query_embedding, 
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document( 
                text=query, 
                model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            ) 
        ],

        query=FusionQuery(fusion="rrf"), 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [34]:
result = retrieve_data(query, k=6)

In [35]:
result

{'retrieved_context_ids': ['B0BN58Z4YX',
  'B0C7DCS2KW',
  'B09RHF4L45',
  'B0B157WDDJ',
  'B0BCFYCXRH',
  'B0BHVH4D37'],
 'retrieved_context': ['ASWINN Tablet Tripod Stand, Gooseneck 65" Height Adjustable Tablet Stand Floor with 360° Rotating Tripod Mount Suitable for iPhone,Tablet,Kindle and All 4.5-12.9 Inch Tablet and Phone (Black)【Free Your Hands】: With this gooseneck arm tablet tripod stand, you can find a more comfortable way to use tablet & Phone in bed or sofa in winter. Release your hands while make your leisure time more enjoyable，the best Christmas gift or New Year gifts for wife,husband,elders,friend or colleague. 【Versatile Gooseneck Tablet Tripod Stand】This ergonomic gooseneck tablet holder suits most tablets and phones. you can easily get your desired angle to meet your needs at different angles and positions. Except for watching movies, you are also allowed to use your tablet for visual presentations, like reading sheet music, watching videos, zoom calls, online teachi

### Reraknking

https://docs.cohere.com/docs/rerank-overview?_gl=1*3sfoj1*_gcl_au*OTkxNzY5OTg2LjE3ODM2MDUwMDUuMTY1NDQwMDMxOC4xNzgzNjA1MDEwLjE3ODM2MDUwMzc.*_ga*MjA4MTAxMDg2OS4xNzgzNjA1MDA1*_ga_CRGS116RZS*czE3ODM2MDUwMDQkbzEkZzEkdDE3ODM2MDY0NzUkajYwJGwwJGgw

In [32]:
cohere_client = cohere.ClientV2()

In [37]:
to_rerank = result["retrieved_context"]

In [38]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20,
)

In [39]:
response

V2RerankResponse(id='5d7a91d3-c48c-4c28-b02f-a6d80c6022b0', results=[V2RerankResponseResultsItem(index=2, relevance_score=0.8846785), V2RerankResponseResultsItem(index=5, relevance_score=0.85994625), V2RerankResponseResultsItem(index=4, relevance_score=0.85565823), V2RerankResponseResultsItem(index=3, relevance_score=0.8517556), V2RerankResponseResultsItem(index=1, relevance_score=0.84877175), V2RerankResponseResultsItem(index=0, relevance_score=0.8270472)], meta=ApiMeta(api_version=ApiMetaApiVersion(version='2', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images=None, input_tokens=None, image_tokens=None, output_tokens=None, search_units=1.0, classifications=None), tokens=None, cached_tokens=None, warnings=None))

this is lighter and more powerful than an llm 

In [40]:
reranked_results = [to_rerank[result.index] for result in response.results]